# Evaluating ProtVS Predictions

This notebook walks through a suite of image quality metrics for comparing
ProtVS-generated protein localization images against real immunofluorescence data.

**Metrics covered:**
| Metric | What it measures |
|---|---|
| Pearson / Spearman R | Pixel-level intensity correlation |
| PSNR | Signal-to-noise ratio (distortion) |
| MS-SSIM | Structural similarity across scales |
| Mutual Information | Shared statistical dependence |
| Haralick (GLCM) | Texture similarity |
| Gabor filter bank | Multi-scale frequency/orientation similarity |
| FID | Distribution-level perceptual similarity (InceptionV3) |

> **Note:** For a full proteome-scale evaluation run, see `evaluate_proteome.py`.
> This notebook uses a small set of example images to illustrate each metric step by step.


## Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from scipy.linalg import sqrtm

from skimage.metrics import structural_similarity
from skimage.feature import graycomatrix, graycoprops
from skimage import filters
from sklearn.metrics import mutual_info_score
from tifffile import imread
from tqdm.auto import tqdm

# Paths — update these to point to your real/predicted TIFF pairs
IMAGE_DIR  = "example_eval_images"   # folder containing *_real.tif and *_pred.tif pairs
RESULTS_DIR = "eval_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


## Step 1: Load Image Pairs

Each example is a matched pair of real and predicted protein channel images.
We extract channel 1 (the protein channel) from each 4-channel TIFF.


In [ ]:
real_files = sorted(f for f in os.listdir(IMAGE_DIR) if "real" in f and f.endswith(".tif"))
print(f"Found {len(real_files)} real images")

pairs = []
for fname in real_files:
    pred_fname = fname.replace("real", "pred")
    pred_path  = os.path.join(IMAGE_DIR, pred_fname)
    if not os.path.exists(pred_path):
        print(f"  Missing prediction for {fname}, skipping")
        continue
    img_real = imread(os.path.join(IMAGE_DIR, fname))[:, :, 1].astype(np.float32)
    img_pred = imread(pred_path)[:, :, 1].astype(np.float32)
    # Skip constant images (undefined correlation / texture)
    if np.std(img_real) == 0 or np.std(img_pred) == 0:
        print(f"  Skipping constant image: {fname}")
        continue
    pairs.append({"name": fname, "real": img_real, "pred": img_pred})

print(f"Loaded {len(pairs)} valid image pairs")


### Visualize an Example Pair

Before computing metrics it's worth visually inspecting a real/predicted pair
to build intuition for what the numbers are capturing.


In [ ]:
if pairs:
    pair = pairs[0]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(pair["real"], cmap="hot"); axes[0].set_title("Real"); axes[0].axis("off")
    axes[1].imshow(pair["pred"], cmap="hot"); axes[1].set_title("Predicted"); axes[1].axis("off")

    diff = pair["real"] - pair["pred"]
    im = axes[2].imshow(diff, cmap="RdBu_r", vmin=-diff.max(), vmax=diff.max())
    axes[2].set_title("Difference (real − pred)"); axes[2].axis("off")
    plt.colorbar(im, ax=axes[2], fraction=0.046)

    plt.suptitle(pair["name"], fontsize=10)
    plt.tight_layout()
    plt.show()


## Step 2: Pixel-Level Correlation

**Pearson R** measures linear correlation between pixel intensities.
**Spearman R** is rank-based and more robust to non-linear monotone relationships.
Both range from −1 to 1; higher is better.


In [ ]:
def pixel_correlations(img_real, img_pred):
    flat_r = img_real.flatten()
    flat_p = img_pred.flatten()
    pearson  = stats.pearsonr(flat_r, flat_p)[0]
    spearman = stats.spearmanr(flat_r, flat_p).statistic
    return pearson, spearman

example = pairs[0]
r, rho = pixel_correlations(example["real"], example["pred"])
print(f"Pearson R : {r:.4f}")
print(f"Spearman R: {rho:.4f}")


## Step 3: PSNR and MS-SSIM

**PSNR** (Peak Signal-to-Noise Ratio) quantifies pixel-level distortion in dB — higher is better,
with ∞ meaning identical images.

**MS-SSIM** (Multi-Scale Structural Similarity) captures luminance, contrast, and structure
at multiple resolutions. Ranges from 0 to 1; higher is better.


In [ ]:
def compute_psnr(img_real, img_pred, data_range=1.0):
    mse = np.mean((img_real - img_pred) ** 2)
    return float('inf') if mse == 0 else 20 * np.log10(data_range) - 10 * np.log10(mse)


def compute_ms_ssim(img_real, img_pred, win_size=11, data_range=1.0):
    weights = np.array([0.0448, 0.2856, 0.3001, 0.2363, 0.1333])
    img1, img2 = img_real.astype(np.float64), img_pred.astype(np.float64)
    mcs, mssim = [], []
    for scale in range(len(weights)):
        sim, cs_map = structural_similarity(img1, img2, win_size=win_size,
                                            data_range=data_range, full=True)
        mssim.append(sim); mcs.append(np.mean(cs_map))
        if scale < len(weights) - 1:
            if min(img1.shape) >= win_size * 2:
                h, w = img1.shape
                img1 = img1[:h//2*2, :w//2*2]; img2 = img2[:h//2*2, :w//2*2]
                img1 = (img1[::2,::2]+img1[1::2,::2]+img1[::2,1::2]+img1[1::2,1::2])/4
                img2 = (img2[::2,::2]+img2[1::2,::2]+img2[::2,1::2]+img2[1::2,1::2])/4
            else:
                weights = weights[:scale+1] / weights[:scale+1].sum(); break
    eps = 1e-8
    mcs = np.clip(mcs, eps, 1.0); mssim = np.clip(mssim, eps, 1.0)
    val = np.prod(mcs[:-1]**weights[:-1]) * mssim[-1]**weights[-1] if len(mcs)>1 else mssim[0]
    return float(np.clip(val, 0.0, 1.0))


psnr_val   = compute_psnr(example["real"], example["pred"])
ms_ssim_val = compute_ms_ssim(example["real"], example["pred"])
print(f"PSNR   : {psnr_val:.2f} dB")
print(f"MS-SSIM: {ms_ssim_val:.4f}")


## Step 4: Mutual Information

Mutual information measures how much knowing one image's pixel values reduces
uncertainty about the other's — capturing non-linear statistical dependence.
Higher is better.


In [ ]:
def compute_mutual_info(img_real, img_pred, bins=256):
    r, p = img_real.flatten(), img_pred.flatten()
    if np.all(r == r[0]) or np.all(p == p[0]):
        return 0.0
    if r.max() > 1.0: r = r / r.max()
    if p.max() > 1.0: p = p / p.max()
    edges = np.linspace(0, 1, bins)
    r_bin = np.clip(np.digitize(r, edges) - 1, 0, bins-1)
    p_bin = np.clip(np.digitize(p, edges) - 1, 0, bins-1)
    return mutual_info_score(r_bin, p_bin)


mi_val = compute_mutual_info(example["real"], example["pred"])
print(f"Mutual Information: {mi_val:.4f} nats")


## Step 5: Texture Similarity (Haralick & Gabor)

Texture metrics measure whether the *spatial patterns* in a prediction match the
real image — important for proteins with structured localizations (e.g., ER networks,
nuclear speckles) that pixel-wise metrics may miss.

- **Haralick (GLCM)** — captures contrast, homogeneity, energy, and correlation
  of gray-level co-occurrences.
- **Gabor filter bank** — decomposes images at multiple spatial frequencies and
  orientations, then correlates the response magnitudes.

Both return an overall similarity in `[0, 1]`; higher is better.


In [ ]:
def haralick_similarity(img1, img2, levels=256):
    def _features(img):
        q = (np.clip(img, 0, 1) * (levels-1)).astype(np.uint8)
        angles_rad = [np.deg2rad(a) for a in [0, 45, 90, 135]]
        glcm = graycomatrix(q, distances=[1], angles=angles_rad,
                            levels=levels, symmetric=True, normed=True)
        return {p: np.mean(graycoprops(glcm, p))
                for p in ['contrast','dissimilarity','homogeneity','energy','correlation','ASM']}
    f1, f2 = _features(img1), _features(img2)
    sims = []
    for k in f1:
        max_v = max(abs(f1[k]), abs(f2[k]), 1e-10)
        sims.append(1.0 - abs(f1[k] - f2[k]) / max_v)
    return float(np.mean(sims))


def gabor_similarity(img1, img2, frequencies=[0.1, 0.3, 0.5], angles=[0, 45, 90, 135]):
    def _responses(img):
        resp = []
        for freq in frequencies:
            for angle in angles:
                real, imag = filters.gabor(img.astype(np.float64), frequency=freq,
                                           theta=np.deg2rad(angle), sigma_x=2, sigma_y=2)
                resp.append(np.sqrt(real**2 + imag**2))
        return resp
    r1, r2 = _responses(img1), _responses(img2)
    corrs = []
    for m1, m2 in zip(r1, r2):
        f1, f2 = m1.flatten(), m2.flatten()
        f1 = (f1 - f1.mean()) / (f1.std() + 1e-10)
        f2 = (f2 - f2.mean()) / (f2.std() + 1e-10)
        c = np.corrcoef(f1, f2)[0, 1]
        corrs.append(c if not np.isnan(c) else 0.0)
    return float(np.mean(corrs))


haralick_score = haralick_similarity(example["real"], example["pred"])
gabor_score    = gabor_similarity(example["real"], example["pred"])
print(f"Haralick similarity: {haralick_score:.4f}")
print(f"Gabor similarity   : {gabor_score:.4f}")


## Step 6: FID — Fréchet Inception Distance

FID compares the **distribution** of InceptionV3 features extracted from real vs. generated
images. Unlike the pairwise metrics above, FID requires a set of images on each side
and rewards realistic-looking diversity — not just closeness to a single reference.

**Lower FID = better.** FID = 0 means the two distributions are identical.

> For a single image pair we compute an L2 feature distance as a proxy.
> The true FID below is computed across all pairs in the example set.


In [ ]:
class InceptionFeatureExtractor:
    def __init__(self, device=DEVICE):
        self.device = device
        self.model = models.inception_v3(
            weights=models.Inception_V3_Weights.DEFAULT, transform_input=False)
        self.model.fc = nn.Identity()
        self.model = self.model.to(device).eval()
        self.preprocess = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(299),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def extract(self, img):
        """img: float [H,W] in [0,1] or uint8; returns 1D feature vector."""
        if img.max() <= 1.0:
            img = (img * 255).astype(np.uint8)
        else:
            img = img.astype(np.uint8)
        rgb = np.stack([img] * 3, axis=-1)  # grayscale → RGB
        with torch.no_grad():
            t = self.preprocess(rgb).unsqueeze(0).to(self.device)
            return self.model(t).cpu().numpy().flatten()


def compute_fid(real_feats, gen_feats):
    mu_r, mu_g = np.mean(real_feats, axis=0), np.mean(gen_feats, axis=0)
    sig_r = np.cov(real_feats, rowvar=False)
    sig_g = np.cov(gen_feats,  rowvar=False)
    diff  = mu_r - mu_g
    covmean, _ = sqrtm(sig_r.dot(sig_g), disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff @ diff + np.trace(sig_r) + np.trace(sig_g) - 2 * np.trace(covmean))


extractor = InceptionFeatureExtractor()
real_feats, gen_feats = [], []
for pair in tqdm(pairs, desc="Extracting InceptionV3 features"):
    real_feats.append(extractor.extract(pair["real"]))
    gen_feats.append(extractor.extract(pair["pred"]))

if len(pairs) > 1:
    fid = compute_fid(np.array(real_feats), np.array(gen_feats))
    print(f"FID (across {len(pairs)} pairs): {fid:.2f}")
else:
    dist = np.linalg.norm(real_feats[0] - gen_feats[0])
    print(f"L2 feature distance (single pair proxy): {dist:.2f}")
    print("  (FID requires multiple images to estimate distributions)")


## Step 7: Compute All Metrics Across All Pairs

Run the full metric suite over every image pair and collect results into a DataFrame.


In [ ]:
records = []
for i, pair in enumerate(tqdm(pairs, desc="Evaluating")):
    r, rho     = pixel_correlations(pair["real"], pair["pred"])
    psnr_v     = compute_psnr(pair["real"], pair["pred"])
    ms_ssim_v  = compute_ms_ssim(pair["real"], pair["pred"])
    mi_v       = compute_mutual_info(pair["real"], pair["pred"])
    haralick_v = haralick_similarity(pair["real"], pair["pred"])
    gabor_v    = gabor_similarity(pair["real"], pair["pred"])
    l2_fid     = np.linalg.norm(real_feats[i] - gen_feats[i])  # per-pair proxy

    records.append({
        "filename":    pair["name"],
        "pearson":     r,
        "spearman":    rho,
        "psnr":        psnr_v,
        "ms_ssim":     ms_ssim_v,
        "mutual_info": mi_v,
        "haralick":    haralick_v,
        "gabor":       gabor_v,
        "l2_fid_proxy": l2_fid,
    })

df = pd.DataFrame(records).set_index("filename")
df.to_csv(os.path.join(RESULTS_DIR, "image_scores.csv"))
print(f"Saved scores → {RESULTS_DIR}/image_scores.csv")
df.describe().round(4)


## Step 8: Visualize the Score Distributions

Box plots give a quick view of how well the model performs across the example set
and which metrics show the most spread.


In [ ]:
metrics_to_plot = ["pearson", "spearman", "psnr", "ms_ssim",
                   "mutual_info", "haralick", "gabor"]
labels = ["Pearson R", "Spearman R", "PSNR (dB)", "MS-SSIM",
          "Mutual Info", "Haralick", "Gabor"]

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(18, 4), sharey=False)
for ax, metric, label in zip(axes, metrics_to_plot, labels):
    ax.boxplot(df[metric].dropna(), patch_artist=True,
               boxprops=dict(facecolor="#aec6e8", color="#2c6fad"),
               medianprops=dict(color="#e74c3c", linewidth=2))
    ax.set_title(label, fontsize=10, fontweight="bold")
    ax.set_xticks([])
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("Metric Distributions Across Example Pairs", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "score_distributions.png"), dpi=150, bbox_inches="tight")
plt.show()


## Step 9: Overall FID

Report the population-level FID score computed from all extracted features.
FID should be interpreted relative to a baseline — for reference, state-of-the-art
generative models on natural images typically achieve FID < 10.


In [ ]:
if len(pairs) > 1:
    print(f"Overall FID: {fid:.2f}")
    with open(os.path.join(RESULTS_DIR, "fid_score.txt"), "w") as f:
        f.write(f"FID: {fid:.4f}\n")
        f.write(f"N pairs: {len(pairs)}\n")
else:
    print("Too few images to compute a meaningful FID. Add more pairs to IMAGE_DIR.")


## Summary

**Outputs** (all under `eval_results/`):
- `image_scores.csv` — per-image scores for all metrics
- `score_distributions.png` — box plot overview
- `fid_score.txt` — overall FID

**Interpreting the metrics:**

Higher is better for Pearson R, Spearman R, PSNR, MS-SSIM, Mutual Information,
Haralick, and Gabor similarity. Lower is better for FID.

Proteins with diffuse cytoplasmic localization tend to score higher on pixel-level
metrics (Pearson, PSNR), while structured compartments (ER, nuclear speckles)
are better captured by texture metrics (Haralick, Gabor) and FID.

For proteome-scale evaluation across thousands of proteins and cell lines,
see the full `evaluate_proteome.py` pipeline.
